# Data/Features

By assigning data to nodes in a ``ToyTree`` you can enable many new approaches for data visualization and statistical analyses.

This section focuses on recommended workflows for assigning data to a tree and fetching the data back in tabular format. We recommend using `ToyTree` objects themselves as the primary data storage object. 

In [1]:
import toytree

# an example tree
tree = toytree.rtree.unittree(ntips=5, seed=123)

## Simple Example
The methods `set_node_data` and `get_node_data` are broadly useful for assigning data to nodes and fetching data from nodes.

The setter method returns a ``ToyTree`` in which ``data`` is assigned to nodes in the tree by matching node queries, such as names, idx labels, or regular expressions.

In [2]:
# dict mapping node queries to values
data = {0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i'}

# store tree with feature "X" set on Nodes 0,1,2
tree = tree.set_node_data(feature="X", data=data)

In [3]:
# get the values of "X" for all Nodes in idx traversal order
tree.get_node_data("X")

0    a
1    b
2    c
3    d
4    e
5    f
6    g
7    h
8    i
Name: X, dtype: object

By returning a ``ToyTree`` you can easily chain the setter method with other tree-based methods to set data to a tree and then subsequently perform steps such as drawing a tree that displays those data, or fetching the data for other analyses. Below we chain the 'set' and 'get' methods in one call.

In [4]:
# chain the two functions together to set & get values for a feature
tree.set_node_data("X", data=data).get_node_data("X")

0    a
1    b
2    c
3    d
4    e
5    f
6    g
7    h
8    i
Name: X, dtype: object

## Features
In `toytree` terminology a "feature" is a named trait stored to one or more nodes in a tree. Each `ToyTree` object has several data features by default: ``idx``, ``name``, ``height``, ``dist``, and ``support``. 

You can create and store additional features under almost any name (except for a few disallowed names and characters). When you load a tree from a newick, NHX, or NEXUS formatted data file it will often contain additional metadata that are stored as features. Several examples are shown in the [tree parsing documentation](parse_trees). 

A `ToyTree` contains a dynamic propery `.features` that lists all features currently assigned one or more Nodes in the tree.

In [5]:
# all feature names assigned to at least one Node in this tree
tree.features

('idx', 'name', 'height', 'dist', 'support', 'X')

### Data as Node attributes

Data stored to a ``ToyTree`` is actually stored to its underlying `Node` objects as [Node attributes](core_node#attributes). 

This is demonstrated below where data is assigned to a feature named "Z" for two Nodes in the tree. Setting and retrieving data directly from Nodes as attributes like this is allowed. However, for general `toytree` usage, we recommend using the helper functions `set_node_data` and `get_node_data` to set and retrieve data as they provide a number of benefits, especially in terms of dealing with missing values, checking data types, and ordering data values.

In [6]:
# set a value for the attribute (feature) named "Z" on two Nodes
tree[0].Z = "A"
tree[1].Z = "B"

When the `get_node_data` function is called without any features selected it returns a dataframe showing all features on the current tree. Here, this tree includes the five default features in addition to the new feature "X" for which we assigned a str value to several Nodes above, and it also includes the attribute "Z", for which we manually assigned values to two Nodes. For other Nodes that do not contain a "Z" feature a default missing value of NaN (math.nan) is returned in the dataframe.

In [7]:
# return a dataframe with all feature data
tree.get_node_data()

,idx,name,height,dist,support,X,Z
0,0,r0,0.0,0.8,NaN,a,A
1,1,r1,0.0,0.4,NaN,b,B
2,2,r2,0.0,0.4,NaN,c,NaN
3,3,r3,0.0,0.8,NaN,d,NaN
4,4,r4,0.0,0.8,NaN,e,NaN
5,5,,0.4,0.4,NaN,f,NaN
6,6,,0.8,0.2,NaN,g,NaN
7,7,,0.8,0.2,NaN,h,NaN
8,8,,1.0,0.0,NaN,i,NaN


## Set Node data
The `set_node_data` method is used to assign data for one feature at a time.

Data can be entered as a mapping (e.g., dictionary) or sequence of values (e.g., list). A number of options are available to make it easier to assign values to many nodes without having to type each name individually. 

A related function is also available, `set_node_data_from_dataframe`, which allows setting multiple features at the same time from tabular data loaded as a pandas DataFrame. Here, however, we will focus on adding single features at a time.

### data: mapping

Enter data as a mapping (e.g., ``dict``) where the keys are valid [Node Queries](data-query), such as a node name, idx label, or regular expression.

You do not have to assign a value to every node in a tree. You can enter ``data`` as a dict selecting only a subset of Nodes and all others will not have their feature value assigned or modified.

In [8]:
# a mapping of node idx labels to values
data = {0: 10, 1: 10, 2: 10, 3: 20, 8: 50}

# set data to feature "Y" for a set of Nodes
tree = tree.set_node_data("Y", data=data)
tree.get_node_data("Y")

0    10.0
1    10.0
2    10.0
3    20.0
4     NaN
5     NaN
6     NaN
7     NaN
8    50.0
Name: Y, dtype: float64

In this example the data dictionary selects nodes using a variety of Node Queries. The first is a regular expression that matches the first four nodes in the tree, the second matches the node named "r4", and the last matches the node with int index of 8. 

In [9]:
# a mapping with different types of supported node query keys
data = {"~r[0-3]": 10, "r4": 20, 8: 50}

# set data to feature "Y" 
tree = tree.set_node_data(feature="Y", data=data)
tree.get_node_data("Y")

0    10.0
1    10.0
2    10.0
3    10.0
4    20.0
5     NaN
6     NaN
7     NaN
8    50.0
Name: Y, dtype: float64

### data: sequence
You can alternatively set data to all Nodes in a tree by entering the values as a sequence (e.g., list, ndarray) in Node idx order. Note that this requires you to have already properly ordered your input data and to be aware of the Node idx order of your current tree. Thus, this method is more error prone than assigning data by dictionary. Nevertheless, the option is available. 

A ``ToyTreeError`` will be raised if you try to set data to a tree using a sequence that is not ``nnodes`` in length. In that case, you must use a ``mapping`` based input such as a dict to specify the assignment.

In [10]:
# get a list of feature vlaues in idx order
data = tree.get_node_data("Y").tolist()

# set data as a sequence of length nnodes in idx order
tree = tree.set_node_data(feature="Y", data=data)
tree.get_node_data("Y")

0    10.0
1    10.0
2    10.0
3    10.0
4    20.0
5     NaN
6     NaN
7     NaN
8    50.0
Name: Y, dtype: float64

### inplace
Use ``inplace=True`` to store data directly to the input tree rather than returning a copy on which the data has been assigned. 

In [11]:
# set data w/ inplace=False returns a copy of the tree with data assigned
tree.set_node_data(feature="Y", data=data, inplace=True)

# set data w/ inplace=True returns the tree with data assigned
tree.set_node_data(feature="Y", data=data, inplace=True);

### default
Use the `default` arg to set a value to all other nodes that do not have assignments in ``data``. 

In [12]:
# set data to feature "Y" using a dict w/ node queries, and the default arg
data = {"~r[0-3]": 10, "r4": 20, 8: 50}
tree.set_node_data(feature="Y", data=data, inplace=True, default=0)
tree.get_node_data("Y")

0    10
1    10
2    10
3    10
4    20
5     0
6     0
7     0
8    50
Name: Y, dtype: int64

Use ``default=float('nan')`` to "clear" the data for a Node that is not listed in ``data``, it will not be overwritten by ``default=None``.

See Also [NaN data](#nan-data).


In [13]:
# set new data to feature "Y" and overwrite previous data at unspecified nodes to NaN
data = {"~r[0-3]": 10, "r4": 20, 8: 50}
tree.set_node_data(feature="Y", data=data, inplace=True, default=float('nan'))
tree.get_node_data("Y")

0    10.0
1    10.0
2    10.0
3    10.0
4    20.0
5     NaN
6     NaN
7     NaN
8    50.0
Name: Y, dtype: float64

### inherit

``inherit`` can be used to assign a value to a node and all of its descendants. 

You can assign values only to the parent node of a clade and use ``inherit=True`` to assign the value to that node and all its descendants. Values are assigned in postorder traversal (roots to tips) so you can enter nested parent node keys.

In [14]:
# set data to feature "Y" for a clade using inherit=True
tree.set_node_data("Y", data={6: 100.0}, inplace=True, inherit=True)
tree.get_node_data("Y")

0    100.0
1    100.0
2    100.0
3     10.0
4     20.0
5    100.0
6    100.0
7      NaN
8     50.0
Name: Y, dtype: float64

## Get Node data

``get_node_data`` extracts feature data from a tree in the correct idx order for plotting. 

Data are returned for a single feature as a pandas ``Series``, or for multiple features as a pandas ``DataFrame``. 

When a feature has not been assigned to all Nodes in a tree ``nan`` will be returned for missing values, but this can be changed by setting a value for the ``missing`` arg.

### Get a single feature
By entering the name of a feature in the tree a pandas Series will be returned with all of the Node values for that feature. Here the Series index contains Node idx labels representing the Nodes in an idxorder traversal of the tree. 

In [15]:
# return values for feature "dist"
tree.get_node_data(feature="dist")

0    0.8
1    0.4
2    0.4
3    0.8
4    0.8
5    0.4
6    0.2
7    0.2
8    0.0
Name: dist, dtype: float64

In [16]:
# return values for feature 'Z' which has data for only 2 Nodes
tree.get_node_data("Z")

0      A
1      B
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
Name: Z, dtype: object

In [17]:
# return values for feature 'Z' with an imputed missing value
tree.get_node_data("Z", missing="C")

0    A
1    B
2    C
3    C
4    C
5    C
6    C
7    C
8    C
Name: Z, dtype: object

The pandas Series object is convenient for viewing and can be easily converted to other object types, like below.

In [20]:
# convert a single trait Series to a dict
tree.get_node_data("Z", missing="C").to_dict()

{0: 'A', 1: 'B', 2: 'C', 3: 'C', 4: 'C', 5: 'C', 6: 'C', 7: 'C', 8: 'C'}

In [21]:
# convert a single trait Series to a numpy ndarray
tree.get_node_data("Z", missing="C").values

array(['A', 'B', 'C', 'C', 'C', 'C', 'C', 'C', 'C'], dtype=object)

### Get multiple features
By default ``get_node_data`` returns a DataFrame with all features in a tree. 


In [22]:
# return Node values for all features
tree.get_node_data()

,idx,name,height,dist,support,X,Y,Z
0,0,r0,0.0,0.8,NaN,a,100.0,A
1,1,r1,0.0,0.4,NaN,b,100.0,B
2,2,r2,0.0,0.4,NaN,c,100.0,NaN
3,3,r3,0.0,0.8,NaN,d,10.0,NaN
4,4,r4,0.0,0.8,NaN,e,20.0,NaN
5,5,,0.4,0.4,NaN,f,100.0,NaN
6,6,,0.8,0.2,NaN,g,100.0,NaN
7,7,,0.8,0.2,NaN,h,NaN,NaN
8,8,,1.0,0.0,NaN,i,50.0,NaN


You can enter a sequence of feature names and of imputed missing values to construct a table for only a subset of features.

In [23]:
# return values for two features, with different imputed missing values
tree.get_node_data(["support", "Z"], missing=[100, "C"])

,support,Z
0,100,A
1,100,B
2,100,C
3,100,C
4,100,C
5,100,C
6,100,C
7,100,C
8,100,C


### NaN data
Across all methods in ``toytree`` missing data is specified by any of the following Python types: ``float('nan')``, ``math.nan``, or ``np.nan`` which all return True by ``np.isnan()``. 

In [24]:
# all objects that return True by np.isnan() are valid missing values in toytree
import numpy as np
import math
print([np.isnan(i) for i in [float('nan'), math.nan, np.nan]])

[True, True, True]


## Using features

See ``range_mapping``, ``color_mapping``, ``annotations``, and many ``pcm`` methods, such as ``simulate``, to see many examples of feature data being used for visualizations or statistical analyses.

## Node vs Edge features
Some data stored to a tree are intended to represent information about the edges (splits) in a tree, rather than information about the nodes. This is important as these types of data must be treated differently when doing things like re-rooting a tree, and in some cases, for visualization. (See the [rooting](/toytree/rooting) tutorial for an example of how this is automatically handled in `toytree`.) Any feature can be optionally plotted as a marker and/or label on edges of a tree rather than on nodes. This can be done in a simple way within the `.draw` function by using the argument `node_as_edge_data=True`, or, it can be done with many more options by using functions in the `toytree.annotate` subpackage.

Examples of plotted edge features are shown below. These have a few key features in visualization: (1) values are plotted on the midpoint of edges; (2) No value is shown for the root edge, since it does not represent a true split in the tree; and (3) only one of the two edges descended from the root show a value, since these are actually the same edge, but on which the root node has been placed. As an example of this last point, a value such as a support score, or edge length, is a feature of this entire edge. Thus, the value is the same whether the tree is rooted or unrooted, as shown below.

In [32]:
# draw a feature as EDGE data
tree.draw(
    node_mask=False,
    node_labels="idx",
    node_labels_style={"font-size": 18},
    node_as_edge_data=True,
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="275.0px" viewBox="0 0 300.0 275.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t619fcf2ecf334ea891306c5b55971696"> 0 1 2 3 4 5 6 r0 r1 r2 r3 r4

In [39]:
# draw a feature as EDGE data for the same tree, unrooted.
tree.unroot().draw(
    node_mask=False,
    node_labels="idx",
    node_labels_style={"font-size": 18},
    node_as_edge_data=True,
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="275.0px" viewBox="0 0 300.0 275.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t00dc3e14f1d6489483e6c7bff36c7143"> 0 1 2 3 4 5 6 r0 r1 r2 r3 r4

Annotation methods can also be used to plot edge data. See the annotation docs.

In [40]:
# annotate a tree with EDGE data
canvas, axes, mark = tree.draw()
tree.annotate.add_edge_labels(axes=axes, labels="idx", font_size=18, mask=False);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="275.0px" viewBox="0 0 300.0 275.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="td2f19578d6fb46b892e6e213116d98c5"> r0 r1 r2 r3 r4 0 1 2 3 4 5 6 7